# 딥러닝 - 비정형 데이터 - 다항분류

## 1. 준비작업
### 1. 패키지 가져오기

In [4]:
from hossam import *

from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
import os

from datetime import datetime as dt
from keras_tuner import Hyperband

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import AUC, F1Score

from tqdm.keras import TqdmCallback

# 로지스틱 성능평가 함수
from sklearn.metrics import confusion_matrix

# 다항 분류의 종속변수 처리를 위한 기능
from tensorflow.keras.utils import to_categorical

# 이미지 로드 기능
from PIL import Image

# 로지스틱 성능평가 함수
from sklearn.metrics import confusion_matrix


In [1]:
!pip install --upgrade "numpy<2.0.0" "pandas<3.0.0" "scipy<1.12.0" "matplotlib<3.8.0"

In [1]:
!pip install -U keras-tuner
!pip install --upgrade numpy

  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hossam 0.5.4 requires numpy<1.27,>=1.21, but you have numpy 2.4.2 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.2 which is incompatible.
matplotlib 3.7.5 requires numpy<2,>=1.20, but you have numpy 2.4.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
esda 2.8.1 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
xarray-einstats 0.10.0 requi

In [2]:
!pip install --upgrade hossam pandas matplotlib

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0

## 2. 데이터셋 준비하기
### 1. 이미지 로드 함수

In [5]:
def load_mnist_png(base_path):

    X = []
    y = []

    for label in sorted(os.listdir(base_path)):
        class_dir = os.path.join(base_path, label)

        if not os.path.isdir(class_dir):
            continue

        for file in os.listdir(class_dir):
            path = os.path.join(class_dir, file)
            img = Image.open(path)
            img_np = np.array(img)

            X.append(img_np)
            y.append(int(label))

    X = np.array(X)
    y = np.array(y)

    return X, y
